# Shenglian Q6/T6/L3 hardware scan: g=0.60

This notebook is one of four matched hardware acquisitions using the same
processed 400-molecule canonical-v3 NPZ, Shenglian qubit chain, W, reservoir
seed, circuit depth, and deterministic label-blind randomized submission
order. Only the global rotation parameter is changed.

`ACTION="submit"` submits 400 circuits. `ACTION="collect"` downloads only
this g point's raw counts and creates `sl_g060_results.zip` for local
feature reconstruction and training.


In [ ]:
# Edit only ACTION, then run all cells. API token is entered securely below.
# "submit": randomized, resumable hardware submission.
# "collect": fetch raw results and create one download ZIP.
ACTION = "submit"  # "submit" or "collect"

from getpass import getpass
API_TOKEN = getpass("New Quafu API token: ")

BACKEND_KEY = 'shenglian'
CHIP = 'Shenglian'
BACKEND_NUM_QUBITS = 84
PHYSICAL_QUBITS = [27, 34, 41, 48, 54, 61]
CALIBRATED_AT = '2026-07-28 19:25:54'
SUBMISSION_ORDER_SEED = 20260728

# Frozen manuscript-mainline circuit.
Q = 6
T = 6
LAYERS = 3
G = 0.60
W = 0.60
RESERVOIR_SEED = 0
REPEAT = 1
DATASET_FILE = 'bace_bal400_canonical_v3_q6_t6.npz'
EXPECTED_DATASET_SHA256 = '00b3fdbab64cb95f61b8580bd7e975ee9f1c9d7df1b44ce60ebe61a643a3eb04'

RESULT_TAG = "sl_g060"
RESULT_ROOT_NAME = f"{RESULT_TAG}_results"
MANIFEST_NAME = f"{RESULT_TAG}_jobs.jsonl"
RAW_DIR_NAME = f"raw_{RESULT_TAG}"
DOWNLOAD_ZIP_NAME = f"{RESULT_TAG}_results.zip"


In [ ]:
import hashlib
import json
import math
import re
import shutil
from pathlib import Path

import numpy as np

PROTOCOL_VERSION = "2026-07-31-shenglian-canonical-v3-q6t6l3-gscan-v1"
RESULT_ROOT = Path(RESULT_ROOT_NAME)
MANIFEST_PATH = RESULT_ROOT / MANIFEST_NAME
RAW_DIR = RESULT_ROOT / RAW_DIR_NAME
ORDER_PATH = RESULT_ROOT / f"{RESULT_TAG}_order.json"
SUMMARY_PATH = RESULT_ROOT / f"{RESULT_TAG}_summary.json"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    return str(value)


def scalar_text(value):
    array = np.asarray(value)
    return str(array.item()) if array.shape == () else str(value)


In [ ]:
def load_processed_dataset():
    path = Path(DATASET_FILE)
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} is missing. Upload the notebook and processed NPZ together."
        )
    actual_hash = sha256_file(path)
    if actual_hash != EXPECTED_DATASET_SHA256:
        raise AssertionError(
            f"Dataset SHA-256 mismatch: {actual_hash} != {EXPECTED_DATASET_SHA256}"
        )
    with np.load(path, allow_pickle=False) as package:
        bundle = {
            "indices": package["indices"].astype(np.int64),
            "tokens": package["tokens"].astype(np.float32),
            "token_hashes": package["token_hashes"].astype(str),
            "schema": scalar_text(package["token_schema_version"]),
            "metadata": json.loads(scalar_text(package["metadata_json"])),
        }
    if bundle["tokens"].shape != (400, T, 3 * Q):
        raise AssertionError(bundle["tokens"].shape)
    if bundle["schema"] != "local-bfs-canonical-isomeric-reparse-stable-v3":
        raise AssertionError(bundle["schema"])
    if bundle["metadata"].get("global_descriptors") is not False:
        raise AssertionError("Global descriptors are forbidden")
    if bundle["metadata"].get("task_guided_traversal") is not False:
        raise AssertionError("Task-guided traversal is forbidden")
    return bundle


BUNDLE = load_processed_dataset()
print(
    {
        "backend": CHIP,
        "dataset_shape": BUNDLE["tokens"].shape,
        "physical_chain": PHYSICAL_QUBITS,
        "calibration_timestamp": CALIBRATED_AT,
    }
)


In [ ]:
def norm_edge(left, right):
    left, right = int(left), int(right)
    return (left, right) if left < right else (right, left)


def baihua_edges():
    edges = set()
    rows, cols = 12, 13
    for row in range(rows):
        for col in range(cols - 1):
            edges.add(norm_edge(row * cols + col, row * cols + col + 1))
    for row in range(rows - 1):
        vertical_cols = (2, 6, 10) if row % 2 == 0 else (0, 4, 8, 12)
        for col in vertical_cols:
            edges.add(norm_edge(row * cols + col, (row + 1) * cols + col))
    return edges


def shenglian_edges():
    # Twelve seven-qubit rows. Even inter-row bands contain both diagonals;
    # odd bands contain the displayed down-left connection only.
    edges = set()
    rows, cols = 12, 7
    for row in range(rows - 1):
        for col in range(cols):
            upper = row * cols + col
            lower_same = (row + 1) * cols + col
            edges.add(norm_edge(upper, lower_same))
            if row % 2 == 0 and col > 0:
                edges.add(norm_edge(upper, (row + 1) * cols + col - 1))
    return edges


def validate_chain():
    if len(PHYSICAL_QUBITS) != Q or len(set(PHYSICAL_QUBITS)) != Q:
        raise AssertionError("Physical chain must contain Q unique qubits")
    if min(PHYSICAL_QUBITS) < 0 or max(PHYSICAL_QUBITS) >= BACKEND_NUM_QUBITS:
        raise AssertionError("Physical chain is outside the backend register")
    coupling = baihua_edges() if BACKEND_KEY == "baihua" else shenglian_edges()
    missing = [
        norm_edge(left, right)
        for left, right in zip(PHYSICAL_QUBITS[:-1], PHYSICAL_QUBITS[1:])
        if norm_edge(left, right) not in coupling
    ]
    if missing:
        raise AssertionError(f"Nonconnected physical chain; missing={missing}")
    return True


def dtc_parameters():
    rng = np.random.default_rng(RESERVOIR_SEED)
    return {
        "J": rng.uniform(-0.9, 0.9, size=Q - 1).astype(np.float32),
        "h": rng.uniform(-W, W, size=Q).astype(np.float32),
    }


def build_circuit(token_sequence):
    from quark.circuit import QuantumCircuit

    validate_chain()
    sequence = np.tanh(np.asarray(token_sequence, dtype=np.float32))
    if sequence.shape != (T, 3 * Q):
        raise AssertionError(sequence.shape)
    params = dtc_parameters()
    qc = QuantumCircuit(BACKEND_NUM_QUBITS, BACKEND_NUM_QUBITS)
    for event in range(T):
        values = sequence[event].reshape(Q, 3)
        for logical, physical in enumerate(PHYSICAL_QUBITS):
            theta, phi, lam = values[logical]
            qc.u(float(theta), float(phi), float(lam), int(physical))
        for _ in range(LAYERS):
            for physical in PHYSICAL_QUBITS:
                qc.rx(G * math.pi, int(physical))
            for start in (0, 1):
                for logical in range(start, Q - 1, 2):
                    left = int(PHYSICAL_QUBITS[logical])
                    right = int(PHYSICAL_QUBITS[logical + 1])
                    qc.cx(left, right)
                    qc.rz(float(params["J"][logical]), right)
                    qc.cx(left, right)
            for logical, physical in enumerate(PHYSICAL_QUBITS):
                qc.rz(float(params["h"][logical]) * math.pi, int(physical))
    qc.measure(PHYSICAL_QUBITS, PHYSICAL_QUBITS)
    metadata = {
        "q": Q,
        "t": T,
        "layers": LAYERS,
        "g": G,
        "W": W,
        "reservoir_seed": RESERVOIR_SEED,
        "J": params["J"].tolist(),
        "h": params["h"].tolist(),
        "interaction": "CX-RZ(J_i)-CX",
        "circuit_qubits": list(PHYSICAL_QUBITS),
        "measured_clbits": list(PHYSICAL_QUBITS),
        "classical_num_bits": BACKEND_NUM_QUBITS,
        "nominal_two_qubit_gates": 2 * T * LAYERS * (Q - 1),
    }
    if metadata["nominal_two_qubit_gates"] != 180:
        raise AssertionError(metadata["nominal_two_qubit_gates"])
    return qc, metadata


validate_chain()
print("Topology validation passed; nominal CX count = 180")


In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def atomic_write_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, default=json_default) + "\n")
    temporary.replace(path)


def task_manager():
    if not API_TOKEN:
        raise RuntimeError("API token is empty")
    try:
        from quark.quafu import Task
    except ImportError:
        from quark import Task
    return Task(API_TOKEN)


def randomized_specs():
    rng = np.random.default_rng(SUBMISSION_ORDER_SEED)
    rows = rng.permutation(len(BUNDLE["indices"])).astype(int)
    specs = [
        {
            "row": int(row),
            "submission_rank": int(rank),
            "original_index": int(BUNDLE["indices"][row]),
        }
        for rank, row in enumerate(rows)
    ]
    RESULT_ROOT.mkdir(parents=True, exist_ok=True)
    ORDER_PATH.write_text(
        json.dumps(
            {
                "protocol_version": PROTOCOL_VERSION,
                "backend": CHIP,
                "result_tag": RESULT_TAG,
                "g": G,
                "seed": SUBMISSION_ORDER_SEED,
                "label_blind_random_permutation": True,
                "original_indices_in_submission_order": [
                    value["original_index"] for value in specs
                ],
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )
    return specs


def submit_all_randomized():
    manager = task_manager()
    existing = read_jsonl(MANIFEST_PATH)
    by_index = {int(record["original_index"]): record for record in existing}
    submitted = 0
    specs = randomized_specs()
    for spec in specs:
        original_index = int(spec["original_index"])
        old = by_index.get(original_index)
        if old is not None and old.get("tid") is not None:
            continue
        row = int(spec["row"])
        qc, metadata = build_circuit(BUNDLE["tokens"][row])
        circuit_id = (
            f"{RESULT_TAG}_rank{spec['submission_rank']:05d}"
            f"_id{original_index:05d}"
        )
        options = {
            "compiler": "quarkcircuit",
            "correct": False,
            "target_qubits": list(PHYSICAL_QUBITS),
        }
        task = {
            "chip": CHIP,
            "name": circuit_id,
            "circuit": qc.to_qlisp,
            "compile": True,
            "correct": False,
            "options": options,
        }
        tid = manager.run(task, repeat=REPEAT)
        record = {
            **spec,
            **metadata,
            "protocol_version": PROTOCOL_VERSION,
            "backend_key": BACKEND_KEY,
            "result_tag": RESULT_TAG,
            "chip": CHIP,
            "physical_qubits": list(PHYSICAL_QUBITS),
            "calibration_timestamp_from_image": CALIBRATED_AT,
            "dataset_file": DATASET_FILE,
            "dataset_sha256": EXPECTED_DATASET_SHA256,
            "token_hash": str(BUNDLE["token_hashes"][row]),
            "circuit_id": circuit_id,
            "tid": str(tid),
            "repeat": REPEAT,
            "status": "Submitted",
            "compile": True,
            "task_options": options,
        }
        by_index[original_index] = record
        ordered = sorted(by_index.values(), key=lambda value: int(value["submission_rank"]))
        atomic_write_jsonl(ordered, MANIFEST_PATH)
        submitted += 1
        print(
            f"[{submitted:03d}] rank={spec['submission_rank']:03d} "
            f"id={original_index} tid={tid}"
        )
    print(f"Submitted {submitted} new jobs; manifest={MANIFEST_PATH}")
    return MANIFEST_PATH


def extract_counts(result):
    if not isinstance(result, dict):
        return None
    for key in ("count", "counts"):
        if result.get(key):
            return result[key]
    for parent in ("data", "result"):
        child = result.get(parent)
        if isinstance(child, dict):
            for key in ("count", "counts"):
                if child.get(key):
                    return child[key]
    return None


def refresh_download_zip():
    archive_base = str(Path(DOWNLOAD_ZIP_NAME).with_suffix(""))
    archive_path = Path(
        shutil.make_archive(
            archive_base,
            "zip",
            root_dir=RESULT_ROOT.parent,
            base_dir=RESULT_ROOT.name,
        )
    )
    print(f"Download this file: {archive_path.resolve()}")
    return archive_path


def collect_raw_results():
    records = read_jsonl(MANIFEST_PATH)
    if not records:
        raise FileNotFoundError(
            f"{MANIFEST_PATH} is missing. Run ACTION='submit' first."
        )
    manager = task_manager()
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    completed = 0
    pending = 0
    failed = 0
    for position, record in enumerate(
        sorted(records, key=lambda value: int(value["submission_rank"]))
    ):
        raw_name = record.get(
            "raw_file_name",
            f"rank{int(record['submission_rank']):05d}"
            f"_id{int(record['original_index']):05d}"
            f"_tid{record['tid']}.json",
        )
        raw_path = RAW_DIR / raw_name
        result = None
        if raw_path.is_file():
            try:
                result = json.loads(raw_path.read_text(encoding="utf-8"))
            except Exception:
                result = None
        if result is None or not extract_counts(result):
            try:
                result = manager.result(record["tid"])
            except Exception as error:
                record["status"] = f"FetchFailed: {error}"
                failed += 1
                atomic_write_jsonl(records, MANIFEST_PATH)
                print(record["circuit_id"], record["status"])
                continue
            raw_path.write_text(
                json.dumps(result, ensure_ascii=False, indent=2, default=json_default),
                encoding="utf-8",
            )
        counts = extract_counts(result)
        record["raw_file_name"] = raw_name
        record["raw_result_path_cloud"] = str(raw_path)
        if counts:
            record["status"] = "CountsDownloaded"
            record["shots_downloaded"] = int(
                round(sum(float(value) for value in counts.values()))
            )
            completed += 1
        else:
            record["status"] = (
                result.get("status")
                or result.get("task_status")
                or result.get("state")
                or "PendingNoCounts"
                if isinstance(result, dict)
                else "PendingNoCounts"
            )
            pending += 1
        atomic_write_jsonl(records, MANIFEST_PATH)
        print(
            f"[{position + 1:03d}/{len(records)}] "
            f"rank={record['submission_rank']} status={record['status']}"
        )
    summary = {
        "protocol_version": PROTOCOL_VERSION,
        "backend": CHIP,
        "physical_qubits": list(PHYSICAL_QUBITS),
        "submitted_records": len(records),
        "counts_downloaded": completed,
        "pending_no_counts": pending,
        "fetch_failed": failed,
        "complete": completed == 400,
        "cloud_processing": "raw result retrieval only; no feature extraction or training",
    }
    SUMMARY_PATH.write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8"
    )
    archive = refresh_download_zip()
    print(json.dumps(summary, indent=2))
    return archive


In [ ]:
if ACTION == "submit":
    OUTPUT = submit_all_randomized()
elif ACTION == "collect":
    OUTPUT = collect_raw_results()
else:
    raise ValueError("ACTION must be 'submit' or 'collect'")

OUTPUT
